In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# ตั้งค่า error mitigation

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';



> **Note:** beta release ของ execution model ใหม่พร้อมใช้งานแล้ว directed execution model ให้ความยืดหยุ่นมากขึ้นในการปรับแต่ง error mitigation workflow ดูข้อมูลเพิ่มเติมใน guide [Directed execution model](/guides/directed-execution-model)


<details>
<summary><b>Package versions</b></summary>

โค้ดในหน้านี้พัฒนาโดยใช้ requirements ต่อไปนี้
แนะนำให้ใช้เวอร์ชันเหล่านี้หรือใหม่กว่า

```
qiskit-ibm-runtime~=0.43.1
```
</details>
เทคนิค error mitigation ช่วยให้ผู้ใช้ลดข้อผิดพลาดของ Circuit โดยการสร้าง model
ของ noise ของอุปกรณ์ในขณะที่รัน โดยทั่วไปแล้วสิ่งนี้ทำให้เกิด overhead
ในการประมวลผลเชิง quantum ที่เกี่ยวข้องกับการ train model และ overhead
ในการประมวลผลเชิง classical เพื่อลดข้อผิดพลาดใน raw results โดยใช้ model
ที่สร้างขึ้น

Estimator primitive รองรับเทคนิค error mitigation หลายอย่าง รวมถึง
[TREX](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#measure_mitigation),
[ZNE](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#zne),
[PEC](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#pec)
และ [PEA](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options)
ดู [Error mitigation and suppression techniques](error-mitigation-and-suppression-techniques)
สำหรับคำอธิบายของแต่ละอย่าง เมื่อใช้ primitives คุณสามารถเปิดหรือปิดวิธีการแต่ละอย่างได้
ดูส่วน [การตั้งค่า error แบบกำหนดเอง](#advanced-error) สำหรับรายละเอียด

> **Note:** Sampler ไม่รองรับ error mitigation แต่คุณสามารถใช้ package [mthree](https://qiskit.github.io/qiskit-addon-mthree/) (matrix-free measurement mitigation) เพื่อทำ error mitigation ในเครื่องได้

Estimator ยังรองรับ `resilience_level` ด้วย resilience level ระบุว่าต้องการสร้าง
ความทนทานต่อข้อผิดพลาดมากแค่ไหน ระดับที่สูงกว่าสร้างผลลัพธ์ที่แม่นยำกว่า แต่ใช้เวลา
ประมวลผลนานกว่า Resilience levels สามารถใช้ตั้งค่า cost/accuracy tradeoff เมื่อใช้
error mitigation กับ primitive query ข้อผิดพลาด mitigation ลดข้อผิดพลาด (bias)
ในผลลัพธ์โดยการประมวลผล outputs จากชุดหรือ ensemble ของ Circuit ที่เกี่ยวข้อง
ระดับของการลดข้อผิดพลาดขึ้นอยู่กับวิธีที่ใช้ resilience level เป็นการ abstract
การเลือกวิธี error mitigation โดยละเอียดเพื่อให้ผู้ใช้สามารถพิจารณา cost/accuracy
tradeoff ที่เหมาะสมกับ application ของตน

เมื่อคำนึงถึงสิ่งนี้ แต่ละระดับจะสอดคล้องกับวิธีหนึ่งหรือหลายวิธีที่มี quantum sampling
overhead เพิ่มขึ้นเพื่อให้คุณทดลองกับ time-accuracy tradeoff ที่แตกต่างกัน
ตารางต่อไปนี้แสดงว่าระดับและวิธีที่สอดคล้องกันใดบ้างที่มีสำหรับแต่ละ primitive

> **Info:** error mitigation เป็น task-specific ดังนั้นเทคนิคที่คุณสามารถใช้จะแตกต่างกันขึ้นอยู่กับว่าคุณกำลัง sampling distribution หรือสร้าง expectation values

<span id="resilience-table"></span>

Estimator รองรับ resilience levels ต่อไปนี้ Sampler ไม่รองรับ resilience levels

| Resilience Level | คำจำกัดความ                                                                                            | เทคนิค                                                                 |
|------------------|-------------------------------------------------------------------------------------------------------|---------------------------------------------------------------------------|
| 0                | ไม่มี mitigation                                                                                         | ไม่มี                                                                      |
| 1 [ค่าเริ่มต้น]      | ต้นทุน mitigation น้อยที่สุด: ลดข้อผิดพลาดที่เกี่ยวข้องกับ readout errors               | Twirled Readout Error eXtinction (TREX) measurement twirling             |
| 2                | ต้นทุน mitigation ปานกลาง โดยทั่วไปลด bias ใน estimators แต่ไม่ได้รับประกันว่าจะ zero-bias | Level 1 + Zero Noise Extrapolation (ZNE) และ gate twirling

> **Info:** Resilience levels ยังอยู่ใน beta ดังนั้น sampling overhead และ solution quality จะแตกต่างกันในแต่ละ Circuit ฟีเจอร์ใหม่ ตัวเลือกขั้นสูง และเครื่องมือจัดการจะเปิดตัวเป็นระยะๆ ไม่ได้รับประกันว่าวิธี error mitigation เฉพาะจะถูกใช้ในแต่ละ resilience level

## ตั้งค่า Estimator ด้วย resilience levels
คุณสามารถใช้ resilience levels เพื่อระบุเทคนิค error mitigation หรือตั้งค่าเทคนิคแบบกำหนดเองแต่ละอย่างตามที่อธิบายใน [การตั้งค่า error แบบกำหนดเอง](#advanced-error)

<details>
<summary>Resilience Level 0</summary>

ไม่มีการใช้ error mitigation กับ user program

</details>

<details>
<summary>Resilience Level 1</summary>

Level 1 ใช้ **readout error mitigation** และ **measurement twirling** โดยการใช้เทคนิค
แบบ model-free ที่เรียกว่า Twirled Readout Error eXtinction (TREX) ซึ่งลด measurement
error โดย diagonalizing noise channel ที่เกี่ยวข้องกับ measurement โดยการสุ่ม flip
Qubit ผ่าน X gates ทันทีก่อนการวัด rescaling term จาก diagonal noise channel ถูก
learn โดยการ benchmark random circuits ที่เริ่มต้นใน zero state สิ่งนี้ช่วยให้ service
สามารถลบ bias ออกจาก expectation values ที่เกิดจาก readout noise แนวทางนี้อธิบาย
เพิ่มเติมใน [Model-free readout-error mitigation for quantum expectation
values](https://arxiv.org/abs/2012.09738)

</details>

<details>
<summary>Resilience Level 2</summary>

Level 2 ใช้ **เทคนิค error mitigation ที่รวมอยู่ใน level 1** และยังใช้ **gate twirling**
และ **Zero Noise Extrapolation method (ZNE)** อีกด้วย ZNE คำนวณ expectation value
ของ observable สำหรับ noise factors ที่แตกต่างกัน (amplification stage) จากนั้นใช้
expectation values ที่วัดได้เพื่อ infer ideal expectation value ที่ zero-noise limit
(extrapolation stage) แนวทางนี้มีแนวโน้มที่จะลดข้อผิดพลาดใน expectation values
แต่ไม่ได้รับประกันว่าจะให้ผลลัพธ์ที่ unbiased

![ภาพนี้แสดง graph แกน x มีป้ายกำกับ Noise amplification factor แกน y มีป้ายกำกับ Expectation value เส้นที่ลาดขึ้นมีป้ายกำกับ Mitigated value จุดใกล้เส้นคือ noise-amplified values มีเส้นแนวนอนเหนือแกน X เล็กน้อยมีป้ายกำกับ Exact value](../docs/images/guides/configure-error-mitigation/resilience-2.svg "Illustration of the ZNE method")

overhead ของวิธีนี้ขยายตามจำนวน noise factors การตั้งค่าเริ่มต้น sample expectation value ที่ noise factors สามค่า ทำให้มี overhead ประมาณ 3 เท่าเมื่อใช้ resilience level นี้

ใน Level 2 วิธี TREX จะสุ่ม flip Qubit ผ่าน X gates ทันทีก่อนการวัด และ flip bit ที่วัดได้ที่สอดคล้องกันถ้ามีการใช้ X gate แนวทางนี้อธิบายเพิ่มเติมใน [Model-free readout-error mitigation for quantum expectation values](https://arxiv.org/abs/2012.09738)

</details>

### ตัวอย่าง
interface `EstimatorV2` ให้ผู้ใช้ทำงานได้อย่างราบรื่นกับวิธี error mitigation หลากหลายเพื่อลดข้อผิดพลาดใน expectation values ของ observables โค้ดต่อไปนี้ใช้ Zero Noise Extrapolation และ readout error mitigation โดยเพียงแค่ตั้งค่า `resilience_level 2`

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during primitive initialization
estimator = Estimator(backend, options={"resilience_level": 2})

<span id="advanced-error"></span>
## การตั้งค่า error แบบกำหนดเอง
คุณสามารถเปิดและปิดวิธี error mitigation และ suppression แต่ละอย่าง รวมถึง dynamical decoupling, gate และ measurement twirling, measurement error mitigation, PEC และ ZNE ดู [Error mitigation and suppression techniques](error-mitigation-and-suppression-techniques) สำหรับคำอธิบายของแต่ละอย่าง

> **Note:** - ไม่ใช่ทุกตัวเลือกที่มีสำหรับทั้งสอง primitives ดู [available options table](runtime-options-overview#options-table) สำหรับรายการตัวเลือกที่มี
> - ไม่ใช่ทุกวิธีที่ทำงานร่วมกันได้บน Circuit ทุกประเภท ดู [feature compatibility table](runtime-options-overview#options-compatibility-table) สำหรับรายละเอียด

<Tabs>
  <TabItem value="Estimator" label="Estimator">
    ```python
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(backend)
options = estimator.options
# Turn on gate twirling.
options.twirling.enable_gates = True
# Turn on measurement error mitigation.
options.resilience.measure_mitigation = True

print(f">>> gate twirling is turned on: {estimator.options.twirling.enable_gates}")
print(f">>> measurement error mitigation is turned on: {estimator.options.resilience.measure_mitigation}")
```
  </TabItem>

  <TabItem value="Sampler" label="Sampler">